# 3. Güvenlik testi

**Setler** (`kod/veri_hatti.py` üretir, `veri/gizli/` altında, depoya girmez):

| Kaynak | İçerik | Ölçtüğü |
|---|---|---|
| HarmBench | 200 zararlı davranış, 6 kategori | reddetme oranı (AS2) |
| Türkiye'ye özgü alt küme | 50 zararlı istem | reddetme oranı (AS2) |
| TR-Kontrol | 44 zararsız ama zararlı görünen istem | aşırı reddetme (AS3) |
| XSTest | çeviriye dayanıklı kategorilerden 46 güvenli istem | aşırı reddetme (AS3) |

XSTest'in `homonyms` kategorisi koşuya girmez: maddeler İngilizcenin çok
anlamlılığına dayanıyor (*execute a plan / execute someone*) ve Türkçede bu
çiftler olmadığı için tuzak kayboluyor. Yerine TR-Kontrol seti kuruldu.

**Etiketleme:** `ortak.classify_response()` cevapları RED / KISMI / UYUM olarak
etiketler. Bu bir ön tarama aracıdır; karar kuralları ve elle doğrulama yöntemi
`veri/etiketleme_rehberi.md` dosyasında. Raporlanan sayılar için `05_analiz.ipynb`
60 cevaplık bir örneklem çıkarır ve Cohen's kappa ile uyumu ölçer.

Üretim `ortak.SAFETY_MAX_TOKENS` (128) token ile sınırlı: reddetme ilk cümlelerde
belli oluyor, sınıflandırma penceresi 300 karakter.

In [ ]:
!pip install -q -U "transformers>=4.43" "accelerate>=0.33" "bitsandbytes>=0.43" "datasets>=2.20"

In [ ]:
import os
import sys
import time

import pandas as pd

# Depo kökünü bul: Colab'da /content/proje ya da /content, Kaggle'da /kaggle/...
for _kok in ["/content/proje", "/content", "/kaggle/working/proje", "/kaggle/working", "."]:
    if os.path.isdir(os.path.join(_kok, "kod")):
        sys.path.insert(0, os.path.join(_kok, "kod"))
        break
else:
    raise RuntimeError(
        "kod/ klasörü bulunamadı. Depoyu klonlayın:\n"
        "  !git clone https://github.com/<kullanici>/<depo>.git /content/proje"
    )

import ortak as O

O.set_seed()
O.hf_login()
print(O.env_report())
print("Çıktı klasörü:", O.OUT_DIR)

In [ ]:
MODEL_KEY = "trendyol"
RUN_VARIANTS = ["int8", "nf4"]

corpus = O.load_safety_corpus()
prompts = O.sample_safety(corpus)
display(prompts.groupby(["kaynak", "tur"]).size().to_frame("istem"))

In [ ]:
tok = O.load_tokenizer(MODEL_KEY)

# Sınıflandırıcının çalıştığını iki örnekle doğrula (gerçek cevap değil, kontrol)
assert O.classify_response("Üzgünüm, bu konuda yardımcı olamam.") == O.RED
assert O.classify_response("Linux'ta kill komutunu kullanabilirsiniz.") == O.UYUM
print("Sınıflandırıcı kontrolü tamam")

In [ ]:
for variant in RUN_VARIANTS:
    run = O.Run(MODEL_KEY, variant, "guvenlik")
    if run.done():
        print(f"{run.name}: zaten var, atlanıyor")
        continue

    print(f"\n===== {MODEL_KEY} / {O.VARIANT_LABELS[variant]} =====")
    O.free_gpu()
    model = O.load_model(MODEL_KEY, variant)

    rows, t0 = [], time.perf_counter()
    for k, (_, r) in enumerate(prompts.iterrows(), 1):
        answer, n_tok, _ = O.generate(tok, model, r["metin_tr"],
                                      max_new_tokens=O.SAFETY_MAX_TOKENS)
        rows.append({"model": MODEL_KEY, "variant": variant, "id": r["id"],
                     "kaynak": r["kaynak"], "kategori": r["kategori"], "tur": r["tur"],
                     "cevap": answer, "uretilen_token": n_tok,
                     "otomatik_etiket": O.classify_response(answer),
                     "insan_etiketi": "", "not": ""})
        if k % 50 == 0:
            red = sum(x["otomatik_etiket"] == O.RED for x in rows) / k
            print(f"  {k}/{len(prompts)}  RED oranı={red:.3f}  "
                  f"{time.perf_counter() - t0:.0f} sn")

    df = pd.DataFrame(rows)
    df.to_csv(run.path(), index=False, encoding="utf-8-sig")
    print(f"Kaydedildi: {run.path()}")

    özet = (df.pivot_table(index="tur", columns="otomatik_etiket",
                           values="id", aggfunc="count", fill_value=0))
    display(özet)

    del model
    O.free_gpu()

## Hatırlatma

`insan_etiketi` ve `not` sütunları boş kaydedildi. `05_analiz.ipynb` içindeki
`etiket_ornegi()` fonksiyonu elle doldurulacak 60 satırlık örneklemi çıkarır.
Özellikle "bozuk üretim" notunu doldurmak önemli: düşük bitlerde reddetme
oranındaki düşüş, modelin daha uyumlu olmasından değil tutarlı metin
üretememesinden kaynaklanabilir.